<center>МИНИСТЕРСТВО НАУКИ И ВЫСШЕГО ОБРАЗОВАНИЯ РОССИЙСКОЙ ФЕДЕРАЦИИ </center>
<center>ФЕДЕРАЛЬНОЕ ГОСУДАРСТВЕННОЕ БЮДЖЕТНОЕ ОБРАЗОВАТЕЛЬНОЕ УЧРЕЖДЕНИЕ ВЫСШЕГО ОБРАЗОВАНИЯ </center>
<center>«НОВОСИБИРСКИЙ ГОСУДАРСТВЕННЫЙ ТЕХНИЧЕСКИЙ УНИВЕРСИТЕТ»</center>
<center>Кафедра Вычислительной техники </center>
<br>
<center> <b> <font size="5">  ОТЧЁТ </font>  </b>  </center>   
<center><font size="3">по лабораторной работе №2</font></center>
<center><font size="3">по дисциплине: «Системы искусственного интеллекта и машинное обучение» </font></center>
<br>

Выполнили:
- Болотенко Н.А.
- Левицкий А.В.

Проверил: Пронюшкина А.Н.

<center>  Новосибирск, 2025  </center>


## Цель работы

Исследование влияния операции приведения данных к новым шкалам (стандартизации) на обучение моделей МО при решении задачи регрессии, с использованием библиотеки scikit-learn.
Знакомство с приемом перекрестной проверки данных (cross-validation).

## Основная часть
---

### Подключение пакетов

Перед началом работы нужно убедиться, что необходимые для работы пакеты установлены в системе.

In [ ]:
import sys
print(f"Версия Python - {sys.version}")
print(f"Путь к интерпретатору Python - {sys.executable}")

In [ ]:
import pandas as pd
import numpy  as np

import sklearn
from sklearn import linear_model
from sklearn import ensemble
from sklearn import metrics
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler

from matplotlib import pyplot as plt
from matplotlib import cm
import seaborn as sns

### Объявление функций 

In [ ]:
def plot_difference(y_test, y_pred) -> None:
    '''
    Функция построения графиков
    :param y_test: - проверочные значения целевой переменной
    :param y_pred: - вычисленные значения целевой переменной
    '''
    plt.figure(figsize=(12,6))
    
    plt.subplot(121)
    plt.scatter(y_test, y_pred,  alpha=0.1, color = "#17becf")
    plt.plot(  [ np.min(y_test), np.max(y_test) ], # x1,x2
               [ np.min(y_test), np.max(y_test) ], # y1,y2
               '--',
               alpha=0.7, lw=3, color = "black")
    plt.title('Диаграмма рассеяния вычисленных значений');
    plt.xlabel('Проверочное Y')
    plt.ylabel('Вычисленное Y')
    plt.grid(True)

    plt.subplot(122) # 1 row, 2 column, 2 index on grid
    plt.scatter(y_test, (y_test - y_pred)**2,  alpha=0.1, color = "#17becf")
    plt.title('Диаграмма рассеяния квадрата абсолютной ошибки')
    plt.xlabel('Проверочное Y')
    plt.ylabel('Квадрат абсолютной ошибки')
    plt.grid(True)

In [ ]:
def stats(y_test, y_pred):
    '''
    Вычисление и вывод метрик: MAE, RMSE, R2. Используются функции из библиотеки sklearn
    На основе сравнения проверочных и вычисленных.
    :param y_test: - проверочные значения целевой переменной
    :param y_pred: - вычисленные значения целевой переменной
    '''
    mae  = metrics.mean_absolute_error(y_test, y_pred)
    mse  = metrics.mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2   = metrics.r2_score(y_test, y_pred)
    print ("MAE : {:>9,.3f} (средняя абсолютная ошибка)".format(mae))
    print ("MSE : {:>9,.6f} (среднеквадратичная ошибка)".format(mse))
    print ("RMSE: {:>9,.6f} (кв. корень из среднеквадратичной ошибки)".format(rmse))
    print ("R2  : {:>9,.3f} (коэфф. детерминации)".format(r2))

### Загрузка подготовленных данных

Структура csv-файла:
- Ячейки разделены ";"
- Дробная часть помечена "."

In [ ]:
import pandas as pd
dataset_file = "datasets/dataset_prepared.csv"
df = pd.read_csv(
    dataset_file,
    sep=';',
    decimal='.',
    header=0
)

df.head()

Выясним размеры датасета

In [ ]:
num_rows, num_cols = df.shape
print(f"Размеры набора данных: {num_rows} строк и {num_cols} столбцов\n")
df.info()

Целевой признак - Median Income: Медианный доход на семью в квартале[10тыс.$].

Коэфф. корреляции Пирсона позволит выбрать независимые переменные.

In [ ]:
target=['Median_Income']
corr_coeffs = df.corr(method='pearson')
corr_coeffs[target[0]].abs().sort_values(ascending=False)

В качестве независимых переменных выберем признаки с высоким абс. значением коэфф. корреляции, но при этом как можно более не связанные между собой. Кандидаты:
- *Median_House_Value* - Медианная цена дома в квартале [$]
- *Distance_to_coast*- расстояние до ближайшей точки побережья [м] (или *OcPrx_INLAND*)
- *Tot_Rooms* - Общее количество комнат в квартале
- *Median_Age* - Медианный возраст дома в квартале; меньше = новее [лет]

In [ ]:
features = ['Median_House_Value', 'Distance_to_coast', 'Tot_Rooms',  'Median_Age']

Уберем все прочие признаки

In [ ]:
df = df[target +features]
df.head()

### Приведение к новой шкале значений

In [ ]:
from sklearn.preprocessing import StandardScaler
scalerStd = StandardScaler()

In [ ]:
scalerStd.fit(df)

In [ ]:
print (" {:>3} {:<25} {:>16} {:>16}".format(
 "№", "Признак", "Средрее", "Ср.кв.откл."
))

for icol in range (0, len(df.columns)):
    print (" {:>3} {:<25} {:>16.3f} {:>16.3f}".format(
                   icol,
                   df.columns[icol],
                   scalerStd.mean_[icol],
                   np.sqrt (scalerStd.var_[icol]), # кв. корень (из дисперсии)
                  ))

In [ ]:
df_scaled = pd.DataFrame (
  data    = scalerStd.transform(df),
  columns = df.columns,
  index   = df.index
)
print("Размер таблицы", df_scaled.shape)
df_scaled.head()

Построим гистограммы распределения целевого признака

In [ ]:
def histograms(df, dfStd, feature_name):
    '''
    Построение гистограмм распределения признака до и после стандартизации
    '''
    plt.figure(figsize=(10,5))

    plt.subplot(121)
    plt.title('Распределение исходных значений')
    plt.xlabel(feature_name)
    plt.ylabel('Количество записей')
    plt.hist(df[feature_name])

    plt.subplot(122)
    plt.title('Распределение стандартизированых значений')
    plt.xlabel(feature_name)
    plt.ylabel('Количество записей')
    plt.hist(dfStd[feature_name])

In [ ]:
histograms(df, df_scaled, target[0])

То же сделаем для независимых признаков

In [ ]:
for feat in features:
    histograms(df, df_scaled, feat)

### Формирование тренировочной и проверочной выборок

Зададим сид генератора случайных чисел

In [ ]:
random_state = 42

А также долю тестовой выборки

In [ ]:
test_size = 0.3

Получим набор на нестандартизированных данных

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(
    df[features],
    df[target],
    test_size=test_size,
    random_state=random_state,
    shuffle=True
)

y_train = y_train[target[0]]
y_test  = y_test[target[0]]

y_test.head()

In [ ]:
print ("Кол-во элементов: \n  x_train: {}, y_train {} \n  x_test:  {}, y_test  {} \n  total x: {}, total y {} ".format  (
    len(x_train), len(y_train),
    len(x_test),  len(y_test),
    len(x_train)+len(x_test), len(y_train)+len(x_test),
))

И на стандартизированных данных

In [ ]:
xStd_train, xStd_test, yStd_train, yStd_test = train_test_split(
    df_scaled[features],
    df_scaled[target],
    test_size=test_size,
    random_state=random_state,
    shuffle=True
)

yStd_train = yStd_train[target[0]]
yStd_test  = yStd_test[target[0]]

In [ ]:
sgd_reg_nonStd = linear_model.SGDRegressor(
    loss='squared_error', # Метод наименьших квадратов (Ordinary least squares)
    max_iter=10000,
    tol=1e-3,
    random_state=8,
)

In [ ]:
sgd_reg_nonStd.fit(x_train,  y_train)

In [ ]:
y_pred = sgd_reg_nonStd.predict(x_test)
y_pred

In [ ]:
plot_difference(y_test, y_pred)

In [ ]:
def hist_diff(y_test, y_pred, bins=20) -> None:
    '''
    Функция построения гистограмм разности и абсолютной разности
    :param y_test: - проверочные значения целевой переменной
    :param y_pred: - вычисленные значения целевой переменной
    '''
    plt.figure(figsize=(10,5))
    
    plt.subplot(121)
    plt.title('Ошибка Y_test - Y_pred')
    plt.xlabel('Значение')
    plt.ylabel('Количество записей')
    plt.hist(y_test - y_pred, bins=bins)
    
    plt.subplot(122)
    plt.title('Ошибка абс. |Y_test - Y_pred|')
    plt.xlabel('Значение')
    plt.ylabel('Количество записей')
    plt.hist(np.abs(y_test - y_pred), bins=bins)

hist_diff(y_test, y_pred, 10)

In [ ]:
stats(y_test, y_pred)

Ситуация гигантских значений ожидаема - [рекомендации](https://scikit-learn.ru/stable/modules/sgd.html#id12:~:text=%D0%A1%D1%82%D0%BE%D1%85%D0%B0%D1%81%D1%82%D0%B8%D1%87%D0%B5%D1%81%D0%BA%D0%B8%D0%B9%20%D0%B3%D1%80%D0%B0%D0%B4%D0%B8%D0%B5%D0%BD%D1%82%D0%BD%D1%8B%D0%B9%20%D1%81%D0%BF%D1%83%D1%81%D0%BA%20%D1%87%D1%83%D0%B2%D1%81%D1%82%D0%B2%D0%B8%D1%82%D0%B5%D0%BB%D0%B5%D0%BD%20%D0%BA%20%D0%BC%D0%B0%D1%81%D1%88%D1%82%D0%B0%D0%B1%D0%B8%D1%80%D0%BE%D0%B2%D0%B0%D0%BD%D0%B8%D1%8E%20%D0%BF%D1%80%D0%B8%D0%B7%D0%BD%D0%B0%D0%BA%D0%BE%D0%B2%2C%20%D0%BF%D0%BE%D1%8D%D1%82%D0%BE%D0%BC%D1%83%20%D0%BD%D0%B0%D1%81%D1%82%D0%BE%D1%8F%D1%82%D0%B5%D0%BB%D1%8C%D0%BD%D0%BE%20%D1%80%D0%B5%D0%BA%D0%BE%D0%BC%D0%B5%D0%BD%D0%B4%D1%83%D0%B5%D1%82%D1%81%D1%8F%20%D0%BC%D0%B0%D1%81%D1%88%D1%82%D0%B0%D0%B1%D0%B8%D1%80%D0%BE%D0%B2%D0%B0%D1%82%D1%8C%20%D0%B2%D0%B0%D1%88%D0%B8%20%D0%B4%D0%B0%D0%BD%D0%BD%D1%8B%D0%B5) указывают, что нужно стандартиировать данные

In [ ]:
sgd_reg_Std = linear_model.SGDRegressor(
    loss='squared_error', # Метод наименьших квадратов (Ordinary least squares)
    max_iter=10000,
    tol=1e-3,
    random_state=8,
)

In [ ]:
sgd_reg_Std.fit(xStd_train,  yStd_train)

In [ ]:
yStd_pred = sgd_reg_Std.predict(xStd_test)
yStd_pred

In [ ]:
plot_difference(yStd_test, yStd_pred)

In [ ]:
hist_diff(yStd_test, yStd_pred, 15)

In [ ]:
stats(yStd_test, yStd_pred)